In [ ]:
# ============================================================
# EXPERIMENT 7 — TOKENS + TF-IDF NEURAL REPRESENTATION FUSION
# ML4CPMS PROJECT 1
#
# Colab-ready single-file notebook script.
#
# Pipeline:
#   Raw token IDs -> CNN -> z_token
#   Token IDs -> TF-IDF -> SVD -> MLP -> z_tfidf
#   [z_token, z_tfidf] -> Fusion MLP -> z_fused
#   z_fused -> binary classifier
#
# Evaluation:
#   1. Neural fusion classifier
#   2. Linear SVM on frozen z_fused
#   3. Logistic Regression on frozen z_fused
#   4. Latent-space geometry
#
# IMPORTANT:
#   - Uses only train.json.
#   - Uses canonical 80/20 stratified split, random_state=42.
#   - Fits TF-IDF/SVD only on the training split.
#   - Does NOT use Kaggle test data.
# ============================================================


# ============================================================


In [ ]:
# CELL 1 — INSTALL / IMPORTS
# ============================================================

!pip -q install scikit-learn pandas numpy scipy

import os
import json
import random
import time
import zipfile
import numpy as np
import pandas as pd

from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using:", DEVICE)


# ============================================================


In [ ]:
# CELL 2 — CONFIGURATION
# ============================================================

SEED = 42

MAX_LEN = 384

# ----------------------------
# Raw-token CNN
# ----------------------------
TOKEN_EMBED_DIM = 128
CNN_CHANNELS = 128
TOKEN_LATENT_DIM = 128

# We use three receptive fields.
KERNELS = (3, 5, 7)

# ----------------------------
# TF-IDF branch
# ----------------------------
TFIDF_MAX_FEATURES = 200_000
TFIDF_MIN_DF = 3
TFIDF_SVD_DIM = 256

# ----------------------------
# Fusion
# ----------------------------
FUSION_HIDDEN_DIM = 256
FUSED_LATENT_DIM = 128

# ----------------------------
# Training
# ----------------------------
BATCH_SIZE = 64
EPOCHS = 15
LEARNING_RATE = 2e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 3

OUTPUT_DIR = Path("/content/exp7_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

print("Seed:", SEED)
print("MAX_LEN:", MAX_LEN)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)


# ============================================================


In [ ]:
# CELL 3 — UPLOAD TRAIN.JSON
# ============================================================

# Run this cell and upload train.json when Colab asks.

from google.colab import files

uploaded = files.upload()

print("\nUploaded:")
for name in uploaded.keys():
    print(" -", name)


# ============================================================


In [ ]:
# CELL 4 — FIND TRAIN.JSON
# ============================================================

# If your file is literally called train.json, this works directly.
# Otherwise, change TRAIN_FILE manually.

TRAIN_FILE = "train.json"

if not os.path.exists(TRAIN_FILE):

    json_files = [
        f for f in os.listdir("/content")
        if f.endswith(".json")
    ]

    if len(json_files) == 1:
        TRAIN_FILE = json_files[0]
        print("Using detected JSON file:", TRAIN_FILE)

    else:
        raise FileNotFoundError(
            "Could not uniquely identify train.json. "
            "Set TRAIN_FILE manually."
        )

print("TRAIN_FILE =", TRAIN_FILE)


# ============================================================


In [ ]:
# CELL 5 — LOAD DATASET
# ============================================================

def load_json_dataset(path):

    with open(path, "r") as f:
        text = f.read().strip()

    # Try standard JSON
    try:

        data = json.loads(text)

        if isinstance(data, list):
            return data

        if isinstance(data, dict):

            for key in [
                "data",
                "train",
                "samples"
            ]:

                if key in data:
                    return data[key]

    except json.JSONDecodeError:
        pass

    # Otherwise JSONL
    data = []

    with open(path, "r") as f:

        for line in f:

            line = line.strip()

            if line:
                data.append(json.loads(line))

    return data


data = load_json_dataset(TRAIN_FILE)

print("Number of documents:", len(data))

print("\nFirst example:")
print(data[0])


# ============================================================


In [ ]:
# CELL 6 — INSPECT DATA FORMAT
# ============================================================

print("\nKeys:")
print(data[0].keys())

print("\nType of text field:")
print(type(data[0]["text"]))

print("\nFirst part of text field:")
print(data[0]["text"][:20])

print("\nLabel:")
print(data[0]["label"])


# ============================================================


In [ ]:
# CELL 7 — EXTRACT TOKENS + LABELS
# ============================================================

TOKEN_KEY = "text"
LABEL_KEY = "label"

sequences = []
labels = []

for row in data:

    seq = row[TOKEN_KEY]
    label = row[LABEL_KEY]

    sequences.append(seq)

    # A = 0
    # B = 1
    if label == "A":

        labels.append(0)

    elif label == "B":

        labels.append(1)

    elif label in [0, 1]:

        labels.append(int(label))

    else:

        raise ValueError(
            f"Unknown label: {label}"
        )


labels = np.asarray(
    labels,
    dtype=np.int64
)

print("Documents:", len(sequences))

print("\nClass distribution:")

unique, counts = np.unique(
    labels,
    return_counts=True
)

for u, c in zip(unique, counts):

    print(
        f"class {u}: {c}"
    )


print("\nSequence length statistics:")

lengths_raw = np.array([
    len(x)
    for x in sequences
])

print("min   :", lengths_raw.min())
print("mean  :", lengths_raw.mean())
print("median:", np.median(lengths_raw))
print("max   :", lengths_raw.max())


# ============================================================


In [ ]:
# CELL 8 — CANONICAL STRATIFIED SPLIT
# ============================================================

indices = np.arange(
    len(sequences)
)

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=SEED,
    stratify=labels
)

print("Train:", len(train_idx))
print("Validation:", len(val_idx))

print("\nTrain class distribution:")
print(
    np.bincount(
        labels[train_idx]
    )
)

print("\nValidation class distribution:")
print(
    np.bincount(
        labels[val_idx]
    )
)


# ============================================================


In [ ]:
# CELL 9 — PREPARE FIXED-LENGTH TOKEN SEQUENCES
# ============================================================

# Find maximum token ID.
max_token = max(
    max(seq) if len(seq) > 0 else 0
    for seq in sequences
)

# Reserve a new ID for padding.
PAD_ID = max_token + 1

VOCAB_SIZE = PAD_ID + 1

print("Maximum token ID:", max_token)
print("PAD ID:", PAD_ID)
print("Vocabulary size:", VOCAB_SIZE)


def prepare_sequences(
    sequences,
    max_len=384,
    pad_id=0
):

    X = np.full(
        (
            len(sequences),
            max_len
        ),
        pad_id,
        dtype=np.int64
    )

    lengths = np.zeros(
        len(sequences),
        dtype=np.int64
    )

    for i, seq in enumerate(sequences):

        if len(seq) == 0:

            # Empty document.
            lengths[i] = 1
            continue

        # Keep the most recent max_len tokens.
        seq = seq[-max_len:]

        n = len(seq)

        X[i, :n] = seq

        lengths[i] = n

    return X, lengths


X_tokens, sequence_lengths = prepare_sequences(
    sequences,
    max_len=MAX_LEN,
    pad_id=PAD_ID
)

print("Token tensor:", X_tokens.shape)
print("Lengths:", sequence_lengths.shape)


# ============================================================


In [ ]:
# CELL 10 — BUILD TF-IDF REPRESENTATION
# ============================================================

# Convert integer token sequences to strings.
#
# Example:
# [12, 45, 17, 45]
#
# becomes:
#
# "12 45 17 45"

documents = [
    " ".join(map(str, seq))
    for seq in sequences
]

train_documents = [
    documents[i]
    for i in train_idx
]

val_documents = [
    documents[i]
    for i in val_idx
]

print("Train documents:", len(train_documents))
print("Validation documents:", len(val_documents))


tfidf = TfidfVectorizer(
    token_pattern=r"(?u)\b\d+\b",
    lowercase=False,
    min_df=TFIDF_MIN_DF,
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=(1, 1),
    sublinear_tf=True
)

print("\nFitting TF-IDF...")

start = time.time()

X_train_tfidf = tfidf.fit_transform(
    train_documents
)

X_val_tfidf = tfidf.transform(
    val_documents
)

tfidf_time = (
    time.time() - start
) / 60

print(
    f"TF-IDF completed in "
    f"{tfidf_time:.2f} min"
)

print(
    "Train TF-IDF:",
    X_train_tfidf.shape
)

print(
    "Val TF-IDF:",
    X_val_tfidf.shape
)


# ============================================================


In [ ]:
# CELL 11 — TF-IDF → SVD
# ============================================================

print("\nRunning TruncatedSVD...")

start = time.time()

svd = TruncatedSVD(
    n_components=TFIDF_SVD_DIM,
    random_state=SEED
)

X_train_svd = svd.fit_transform(
    X_train_tfidf
)

X_val_svd = svd.transform(
    X_val_tfidf
)

svd_time = (
    time.time() - start
) / 60

print(
    f"SVD completed in "
    f"{svd_time:.2f} min"
)

print(
    "Train SVD:",
    X_train_svd.shape
)

print(
    "Val SVD:",
    X_val_svd.shape
)

print(
    "Explained variance ratio:",
    svd.explained_variance_ratio_.sum()
)


# ============================================================


In [ ]:
# CELL 12 — PYTORCH DATASET
# ============================================================

class FusionDataset(Dataset):

    def __init__(
        self,
        token_data,
        tfidf_data,
        labels,
        lengths
    ):

        self.tokens = torch.tensor(
            token_data,
            dtype=torch.long
        )

        self.tfidf = torch.tensor(
            tfidf_data,
            dtype=torch.float32
        )

        self.labels = torch.tensor(
            labels,
            dtype=torch.float32
        )

        self.lengths = torch.tensor(
            lengths,
            dtype=torch.long
        )

    def __len__(self):

        return len(self.labels)

    def __getitem__(self, idx):

        return {
            "tokens": self.tokens[idx],
            "tfidf": self.tfidf[idx],
            "length": self.lengths[idx],
            "label": self.labels[idx]
        }


train_dataset = FusionDataset(
    X_tokens[train_idx],
    X_train_svd,
    labels[train_idx],
    sequence_lengths[train_idx]
)

val_dataset = FusionDataset(
    X_tokens[val_idx],
    X_val_svd,
    labels[val_idx],
    sequence_lengths[val_idx]
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Val batches:",
    len(val_loader)
)


# ============================================================


In [ ]:
# CELL 13 — TOKEN CNN ENCODER
# ============================================================

class TokenCNNEncoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        pad_id,
        embed_dim=128,
        channels=128,
        latent_dim=128,
        kernels=(3, 5, 7)
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=pad_id
        )

        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embed_dim,
                out_channels=channels,
                kernel_size=k
            )
            for k in kernels
        ])

        self.projection = nn.Sequential(

            nn.Linear(
                len(kernels) * channels,
                latent_dim
            ),

            nn.LayerNorm(
                latent_dim
            ),

            nn.GELU()
        )

    def forward(self, tokens):

        # (B, T)
        x = self.embedding(tokens)

        # (B, T, E)
        # ->
        # (B, E, T)
        x = x.permute(
            0,
            2,
            1
        )

        conv_outputs = []

        for conv in self.convs:

            # Convolution
            c = conv(x)

            # Activation
            c = nn.functional.gelu(c)

            # Global max pooling
            c = nn.functional.max_pool1d(
                c,
                c.shape[2]
            ).squeeze(2)

            conv_outputs.append(c)

        # Concatenate all kernel representations.
        x = torch.cat(
            conv_outputs,
            dim=1
        )

        # Project to latent vector.
        z = self.projection(x)

        return z


# ============================================================


In [ ]:
# CELL 14 — TF-IDF ENCODER
# ============================================================

class TfidfEncoder(nn.Module):

    def __init__(
        self,
        input_dim=256,
        latent_dim=128
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                256
            ),

            nn.LayerNorm(256),

            nn.GELU(),

            nn.Dropout(0.2),

            nn.Linear(
                256,
                latent_dim
            ),

            nn.LayerNorm(
                latent_dim
            ),

            nn.GELU()
        )

    def forward(self, x):

        return self.network(x)


# ============================================================


In [ ]:
# CELL 15 — TOKENS + TF-IDF FUSION MODEL
# ============================================================

class TokenTfidfFusionModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        pad_id,
        tfidf_dim=256,
        token_latent_dim=128,
        fused_latent_dim=128
    ):

        super().__init__()

        # ----------------------------------------------------
        # Branch 1: raw token CNN
        # ----------------------------------------------------

        self.token_encoder = TokenCNNEncoder(
            vocab_size=vocab_size,
            pad_id=pad_id,
            embed_dim=TOKEN_EMBED_DIM,
            channels=CNN_CHANNELS,
            latent_dim=token_latent_dim,
            kernels=KERNELS
        )

        # ----------------------------------------------------
        # Branch 2: TF-IDF
        # ----------------------------------------------------

        self.tfidf_encoder = TfidfEncoder(
            input_dim=tfidf_dim,
            latent_dim=token_latent_dim
        )

        # ----------------------------------------------------
        # Fusion
        # ----------------------------------------------------

        self.fusion = nn.Sequential(

            nn.Linear(
                token_latent_dim * 2,
                FUSION_HIDDEN_DIM
            ),

            nn.LayerNorm(
                FUSION_HIDDEN_DIM
            ),

            nn.GELU(),

            nn.Dropout(0.25),

            nn.Linear(
                FUSION_HIDDEN_DIM,
                fused_latent_dim
            ),

            nn.LayerNorm(
                fused_latent_dim
            ),

            nn.GELU()
        )

        # ----------------------------------------------------
        # Classification head
        # ----------------------------------------------------

        self.classifier = nn.Linear(
            fused_latent_dim,
            1
        )

    def forward(
        self,
        tokens,
        tfidf
    ):

        # Raw-token representation
        z_token = self.token_encoder(
            tokens
        )

        # TF-IDF representation
        z_tfidf = self.tfidf_encoder(
            tfidf
        )

        # Concatenate representations
        combined = torch.cat(
            [
                z_token,
                z_tfidf
            ],
            dim=1
        )

        # Learned fused representation
        z_fused = self.fusion(
            combined
        )

        # Classification
        logits = self.classifier(
            z_fused
        )

        return (
            logits.squeeze(1),
            z_fused
        )


# ============================================================


In [ ]:
# CELL 16 — CREATE MODEL
# ============================================================

model = TokenTfidfFusionModel(
    vocab_size=VOCAB_SIZE,
    pad_id=PAD_ID,
    tfidf_dim=TFIDF_SVD_DIM,
    token_latent_dim=TOKEN_LATENT_DIM,
    fused_latent_dim=FUSED_LATENT_DIM
).to(DEVICE)


num_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(model)

print(
    "\nTrainable parameters:",
    f"{num_params:,}"
)


# ============================================================


In [ ]:
# CELL 17 — LOSS + OPTIMIZER
# ============================================================

n_class0 = np.sum(
    labels[train_idx] == 0
)

n_class1 = np.sum(
    labels[train_idx] == 1
)

pos_weight = (
    n_class0 / n_class1
)

print("Class 0:", n_class0)
print("Class 1:", n_class1)
print("Positive weight:", pos_weight)


criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        pos_weight,
        dtype=torch.float32,
        device=DEVICE
    )
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# ============================================================


In [ ]:
# CELL 18 — EVALUATION FUNCTION
# ============================================================

@torch.no_grad()
def evaluate(model, loader):

    model.eval()

    all_labels = []
    all_probs = []
    all_z = []

    total_loss = 0.0
    total_examples = 0

    for batch in loader:

        tokens = batch["tokens"].to(
            DEVICE
        )

        tfidf = batch["tfidf"].to(
            DEVICE
        )

        labels_batch = batch["label"].to(
            DEVICE
        )

        logits, z = model(
            tokens,
            tfidf
        )

        loss = criterion(
            logits,
            labels_batch
        )

        probs = torch.sigmoid(
            logits
        )

        batch_size = len(
            labels_batch
        )

        total_loss += (
            loss.item() * batch_size
        )

        total_examples += batch_size

        all_labels.append(
            labels_batch
            .cpu()
            .numpy()
        )

        all_probs.append(
            probs
            .cpu()
            .numpy()
        )

        all_z.append(
            z
            .cpu()
            .numpy()
        )

    y = np.concatenate(
        all_labels
    )

    probs = np.concatenate(
        all_probs
    )

    z = np.concatenate(
        all_z
    )

    predictions = (
        probs >= 0.5
    ).astype(int)

    accuracy = accuracy_score(
        y,
        predictions
    )

    auc = roc_auc_score(
        y,
        probs
    )

    loss = (
        total_loss /
        total_examples
    )

    return {
        "loss": loss,
        "accuracy": accuracy,
        "auc": auc,
        "y": y,
        "probs": probs,
        "z": z
    }


# ============================================================


In [ ]:
# CELL 19 — TRAINING
# ============================================================

best_val_loss = float("inf")

best_state = None

patience_counter = 0

history = []

start_training = time.time()


for epoch in range(
    1,
    EPOCHS + 1
):

    model.train()

    running_loss = 0.0
    total = 0

    for batch in train_loader:

        tokens = batch["tokens"].to(
            DEVICE
        )

        tfidf = batch["tfidf"].to(
            DEVICE
        )

        y = batch["label"].to(
            DEVICE
        )

        optimizer.zero_grad()

        logits, z = model(
            tokens,
            tfidf
        )

        loss = criterion(
            logits,
            y
        )

        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        batch_size = len(y)

        running_loss += (
            loss.item() *
            batch_size
        )

        total += batch_size

    train_loss = (
        running_loss /
        total
    )

    # Validation
    val_result = evaluate(
        model,
        val_loader
    )

    val_loss = val_result["loss"]
    val_acc = val_result["accuracy"]
    val_auc = val_result["auc"]

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "val_auc": val_auc
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss {train_loss:.4f} | "
        f"Val Loss {val_loss:.4f} | "
        f"Val Acc {val_acc*100:.3f}% | "
        f"Val AUC {val_auc:.4f}"
    )

    # --------------------------------------------------------
    # Save best model
    # --------------------------------------------------------

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }

        patience_counter = 0

        print(
            "  -> saved new best model"
        )

    else:

        patience_counter += 1

        print(
            f"  -> no improvement "
            f"({patience_counter}/{PATIENCE})"
        )

        if patience_counter >= PATIENCE:

            print(
                "\nEarly stopping."
            )

            break


training_time = (
    time.time() -
    start_training
) / 60

print(
    f"\nTraining completed in "
    f"{training_time:.2f} minutes"
)


# ============================================================


In [ ]:
# CELL 20 — RESTORE BEST MODEL
# ============================================================

if best_state is None:

    raise RuntimeError(
        "No best model state was saved."
    )

model.load_state_dict(
    best_state
)

model.to(DEVICE)

best_val_result = evaluate(
    model,
    val_loader
)

print(
    "Best validation accuracy:",
    f"{best_val_result['accuracy']*100:.4f}%"
)

print(
    "Best validation AUC:",
    f"{best_val_result['auc']:.4f}"
)

print(
    "Best validation loss:",
    f"{best_val_result['loss']:.4f}"
)


# ============================================================


In [ ]:
# CELL 21 — CONFUSION MATRIX
# ============================================================

y_val = best_val_result["y"]

p_val = best_val_result["probs"]

pred_val = (
    p_val >= 0.5
).astype(int)

print(
    confusion_matrix(
        y_val,
        pred_val
    )
)

print()

print(
    classification_report(
        y_val,
        pred_val,
        digits=4
    )
)


# ============================================================


In [ ]:
# CELL 22 — EXTRACT LEARNED REPRESENTATIONS
# ============================================================

train_result = evaluate(
    model,
    train_loader
)

val_result = evaluate(
    model,
    val_loader
)

z_train = train_result["z"]
z_val = val_result["z"]

y_train = train_result["y"].astype(int)
y_val = val_result["y"].astype(int)

print(
    "Train embedding:",
    z_train.shape
)

print(
    "Val embedding:",
    z_val.shape
)


# ============================================================


In [ ]:
# CELL 23 — LINEAR SVM ON z_fused
# ============================================================

scaler = StandardScaler()

z_train_scaled = scaler.fit_transform(
    z_train
)

z_val_scaled = scaler.transform(
    z_val
)


svm = LinearSVC(
    C=1.0,
    class_weight="balanced",
    max_iter=10000,
    random_state=SEED
)

svm.fit(
    z_train_scaled,
    y_train
)

svm_pred = svm.predict(
    z_val_scaled
)

svm_score = svm.decision_function(
    z_val_scaled
)

svm_acc = accuracy_score(
    y_val,
    svm_pred
)

svm_auc = roc_auc_score(
    y_val,
    svm_score
)

print(
    "SVM on z_fused"
)

print(
    "Accuracy:",
    f"{svm_acc*100:.4f}%"
)

print(
    "AUC:",
    f"{svm_auc:.4f}"
)


# ============================================================


In [ ]:
# CELL 24 — LOGISTIC REGRESSION ON z_fused
# ============================================================

lr = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    max_iter=3000,
    random_state=SEED
)

lr.fit(
    z_train_scaled,
    y_train
)

lr_pred = lr.predict(
    z_val_scaled
)

lr_prob = lr.predict_proba(
    z_val_scaled
)[:, 1]

lr_acc = accuracy_score(
    y_val,
    lr_pred
)

lr_auc = roc_auc_score(
    y_val,
    lr_prob
)

print(
    "Logistic Regression on z_fused"
)

print(
    "Accuracy:",
    f"{lr_acc*100:.4f}%"
)

print(
    "AUC:",
    f"{lr_auc:.4f}"
)


# ============================================================


In [ ]:
# CELL 25 — LATENT SPACE GEOMETRY
# ============================================================

def centroid_distance(
    z,
    y
):

    z0 = z[y == 0]
    z1 = z[y == 1]

    c0 = z0.mean(axis=0)
    c1 = z1.mean(axis=0)

    return np.linalg.norm(
        c0 - c1
    )


def within_class_distance(
    z,
    y,
    n_pairs=10000,
    seed=42
):

    rng = np.random.default_rng(
        seed
    )

    distances = []

    for cls in [0, 1]:

        zc = z[y == cls]

        n = len(zc)

        i = rng.integers(
            0,
            n,
            size=n_pairs
        )

        j = rng.integers(
            0,
            n,
            size=n_pairs
        )

        d = np.linalg.norm(
            zc[i] - zc[j],
            axis=1
        )

        distances.extend(d)

    return float(
        np.mean(distances)
    )


between = centroid_distance(
    z_val,
    y_val
)

within = within_class_distance(
    z_val,
    y_val
)

ratio = (
    between /
    within
)

print(
    "Latent-space diagnostics"
)

print(
    "-------------------------"
)

print(
    "Between-class centroid distance:",
    f"{between:.4f}"
)

print(
    "Within-class average distance:",
    f"{within:.4f}"
)

print(
    "Between / within ratio:",
    f"{ratio:.4f}"
)


# ============================================================


In [ ]:
# CELL 26 — FINAL EXPERIMENT SUMMARY
# ============================================================

summary = pd.DataFrame([

    {
        "Model":
            "Neural Fusion Classifier",

        "Accuracy":
            best_val_result["accuracy"],

        "AUC":
            best_val_result["auc"]
    },

    {
        "Model":
            "Linear SVM on z_fused",

        "Accuracy":
            svm_acc,

        "AUC":
            svm_auc
    },

    {
        "Model":
            "Logistic Regression on z_fused",

        "Accuracy":
            lr_acc,

        "AUC":
            lr_auc
    }
])

summary["Accuracy (%)"] = (
    summary["Accuracy"] * 100
)

display(
    summary[
        [
            "Model",
            "Accuracy (%)",
            "AUC"
        ]
    ]
)

print()

print(
    "Latent dimension:",
    FUSED_LATENT_DIM
)

print(
    "Between/within ratio:",
    f"{ratio:.4f}"
)


# ============================================================


In [ ]:
# CELL 27 — SAVE MODEL + EMBEDDINGS + REPORT
# ============================================================

# Save model
torch.save(

    {
        "model_state_dict":
            model.state_dict(),

        "config": {

            "seed": SEED,

            "max_len":
                MAX_LEN,

            "embed_dim":
                TOKEN_EMBED_DIM,

            "cnn_channels":
                CNN_CHANNELS,

            "kernels":
                KERNELS,

            "token_latent_dim":
                TOKEN_LATENT_DIM,

            "tfidf_max_features":
                TFIDF_MAX_FEATURES,

            "tfidf_min_df":
                TFIDF_MIN_DF,

            "tfidf_svd_dim":
                TFIDF_SVD_DIM,

            "fused_latent_dim":
                FUSED_LATENT_DIM
        }
    },

    OUTPUT_DIR /
    "exp7_tokens_tfidf_model.pt"
)


# Save embeddings
np.savez_compressed(

    OUTPUT_DIR /
    "exp7_tokens_tfidf_embeddings.npz",

    z_train=z_train,

    z_val=z_val,

    y_train=y_train,

    y_val=y_val
)


# Save training history
pd.DataFrame(
    history
).to_csv(

    OUTPUT_DIR /
    "training_history.csv",

    index=False
)


# Save report
experiment_report = {

    "experiment":
        "Exp7 Tokens + TF-IDF",

    "seed":
        SEED,

    "train_size":
        len(train_idx),

    "val_size":
        len(val_idx),

    "max_len":
        MAX_LEN,

    "tfidf_max_features":
        TFIDF_MAX_FEATURES,

    "tfidf_min_df":
        TFIDF_MIN_DF,

    "tfidf_svd_dim":
        TFIDF_SVD_DIM,

    "token_latent_dim":
        TOKEN_LATENT_DIM,

    "fused_latent_dim":
        FUSED_LATENT_DIM,

    "neural_accuracy":
        float(
            best_val_result["accuracy"]
        ),

    "neural_auc":
        float(
            best_val_result["auc"]
        ),

    "svm_accuracy":
        float(
            svm_acc
        ),

    "svm_auc":
        float(
            svm_auc
        ),

    "lr_accuracy":
        float(
            lr_acc
        ),

    "lr_auc":
        float(
            lr_auc
        ),

    "between_distance":
        float(
            between
        ),

    "within_distance":
        float(
            within
        ),

    "between_within_ratio":
        float(
            ratio
        ),

    "num_parameters":
        int(
            num_params
        ),

    "tfidf_minutes":
        float(
            tfidf_time
        ),

    "svd_minutes":
        float(
            svd_time
        ),

    "training_minutes":
        float(
            training_time
        )
}


with open(
    OUTPUT_DIR /
    "exp7_report.json",
    "w"
) as f:

    json.dump(
        experiment_report,
        f,
        indent=2
    )


print(
    "\nSaved files:"
)

for p in sorted(
    OUTPUT_DIR.iterdir()
):

    print(
        " -",
        p.name
    )


# ============================================================


In [ ]:
# CELL 28 — ZIP OUTPUTS
# ============================================================

zip_path = Path(
    "/content/exp7_outputs.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zf:

    for file_path in OUTPUT_DIR.iterdir():

        if file_path.is_file():

            zf.write(
                file_path,
                arcname=file_path.name
            )

print(
    "\nCreated:",
    zip_path
)


# ============================================================
# FINAL NUMBERS — EASY TO COPY INTO TEAM TRACKER
# ============================================================

print("\n")
print("=" * 60)
print("EXP 7 FINAL RESULTS")
print("=" * 60)

print(
    f"Neural accuracy : "
    f"{best_val_result['accuracy']*100:.4f}%"
)

print(
    f"Neural AUC      : "
    f"{best_val_result['auc']:.4f}"
)

print(
    f"SVM accuracy    : "
    f"{svm_acc*100:.4f}%"
)

print(
    f"SVM AUC         : "
    f"{svm_auc:.4f}"
)

print(
    f"LR accuracy     : "
    f"{lr_acc*100:.4f}%"
)

print(
    f"LR AUC          : "
    f"{lr_auc:.4f}"
)

print(
    f"Between distance: "
    f"{between:.4f}"
)

print(
    f"Within distance : "
    f"{within:.4f}"
)

print(
    f"B/W ratio       : "
    f"{ratio:.4f}"
)

print(
    f"Parameters      : "
    f"{num_params:,}"
)

print(
    f"Training time   : "
    f"{training_time:.2f} min"
)

print("=" * 60)


In [ ]:
# ============================================================
# FINAL TEST PIPELINE — CELL 29
# Upload test.json + sample_submission.csv
#
# The project specification requires:
#   - 3,000 test predictions
#   - columns exactly: id,label
#   - IDs in the same order as test.json
# ============================================================

from google.colab import files
import os
import json
import numpy as np
import pandas as pd

print("Upload BOTH test.json and sample_submission.csv.")
print("If you do not have sample_submission.csv, you may upload only test.json;")
print("the next cell can construct the submission format itself.")

uploaded_test = files.upload()

print("\nUploaded:")
for name in uploaded_test.keys():
    print(" -", name)

TEST_FILE = "test.json"
SAMPLE_SUBMISSION_FILE = "sample_submission.csv"

if not os.path.exists(TEST_FILE):
    json_candidates = [
        f for f in os.listdir("/content")
        if f.endswith(".json") and f != TRAIN_FILE
    ]
    if len(json_candidates) == 1:
        TEST_FILE = json_candidates[0]
    else:
        raise FileNotFoundError(
            "Could not uniquely identify test.json. "
            "Set TEST_FILE manually."
        )

print("\nUsing TEST_FILE:", TEST_FILE)

if os.path.exists(SAMPLE_SUBMISSION_FILE):
    print("Using sample submission:", SAMPLE_SUBMISSION_FILE)
else:
    print("sample_submission.csv not found; it will be constructed from test IDs.")


In [ ]:
# ============================================================
# FINAL TEST PIPELINE — CELL 30
#
# FULL-DATA TRAINING -> TEST PREDICTION -> submission.csv
#
# This cell deliberately retrains from scratch on ALL 10,536
# labelled training documents.
#
# We do NOT use the 80/20 validation split for fitting the final
# model. We only use the validation experiment to determine the
# training duration and (optionally) the classification threshold.
# ============================================================

import time
import json
import numpy as np
import pandas as pd
import torch

from torch.utils.data import TensorDataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import accuracy_score


# ============================================================
# 1. LOAD TEST DATA
# ============================================================

def load_unlabelled_json(path):

    records = []

    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    ids = [r["id"] for r in records]
    texts = [r["text"] for r in records]

    return ids, texts


test_ids, test_sequences = load_unlabelled_json(
    TEST_FILE
)

print("Test documents:", len(test_sequences))

assert len(test_sequences) == 3000, (
    f"Expected 3000 test documents, "
    f"got {len(test_sequences)}"
)

assert len(set(test_ids)) == len(test_ids), (
    "Test IDs are not unique."
)


# ============================================================
# 2. RECREATE FULL TRAINING DOCUMENTS
# ============================================================

# 'sequences' and 'labels' came from the training data loaded
# earlier in this notebook.

all_train_documents = [
    " ".join(map(str, seq))
    for seq in sequences
]

all_test_documents = [
    " ".join(map(str, seq))
    for seq in test_sequences
]

print(
    "Full training documents:",
    len(all_train_documents)
)

print(
    "Test documents:",
    len(all_test_documents)
)


# ============================================================
# 3. FIT TF-IDF ON ALL TRAINING DATA
# ============================================================
#
# IMPORTANT:
# The test set is NEVER used to fit TF-IDF.
# We only transform test documents using the vocabulary/weights
# learned from the labelled training set.
# ============================================================

print("\nFitting FULL-DATA TF-IDF...")

start = time.time()

tfidf_full = TfidfVectorizer(
    token_pattern=r"(?u)\b\d+\b",
    lowercase=False,
    min_df=TFIDF_MIN_DF,
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=(1, 1),
    sublinear_tf=True
)

X_full_tfidf = tfidf_full.fit_transform(
    all_train_documents
)

X_test_tfidf = tfidf_full.transform(
    all_test_documents
)

tfidf_full_time = (
    time.time() - start
) / 60

print(
    f"Full TF-IDF completed in "
    f"{tfidf_full_time:.2f} min"
)

print(
    "Full train TF-IDF:",
    X_full_tfidf.shape
)

print(
    "Test TF-IDF:",
    X_test_tfidf.shape
)


# ============================================================
# 4. FIT SVD ON ALL TRAINING DATA
# ============================================================

print("\nFitting FULL-DATA SVD...")

start = time.time()

svd_full = TruncatedSVD(
    n_components=TFIDF_SVD_DIM,
    random_state=SEED
)

X_full_svd = svd_full.fit_transform(
    X_full_tfidf
)

X_test_svd = svd_full.transform(
    X_test_tfidf
)

svd_full_time = (
    time.time() - start
) / 60

print(
    f"Full SVD completed in "
    f"{svd_full_time:.2f} min"
)

print(
    "Full train SVD:",
    X_full_svd.shape
)

print(
    "Test SVD:",
    X_test_svd.shape
)

print(
    "Explained variance:",
    svd_full.explained_variance_ratio_.sum()
)


# ============================================================
# 5. PREPARE FULL TRAIN + TEST TOKEN SEQUENCES
# ============================================================

X_full_tokens, full_lengths = prepare_sequences(
    sequences,
    max_len=MAX_LEN,
    pad_id=PAD_ID
)

X_test_tokens, test_lengths = prepare_sequences(
    test_sequences,
    max_len=MAX_LEN,
    pad_id=PAD_ID
)

print(
    "Full train tokens:",
    X_full_tokens.shape
)

print(
    "Test tokens:",
    X_test_tokens.shape
)


# ============================================================
# 6. DETERMINE FINAL NUMBER OF TRAINING EPOCHS
# ============================================================
#
# We already performed the proper 80/20 validation experiment.
# Find the epoch with the lowest validation loss and use that
# training duration for the final full-data model.
#
# This avoids using the hidden test labels to choose epochs.
# ============================================================

history_df = pd.DataFrame(history)

best_epoch_row = history_df.loc[
    history_df["val_loss"].idxmin()
]

FINAL_EPOCHS = int(
    best_epoch_row["epoch"]
)

print(
    "\nBest validation-loss epoch:",
    FINAL_EPOCHS
)

print(
    "Final model will train for",
    FINAL_EPOCHS,
    "epochs on ALL labelled data."
)


# ============================================================
# 7. DETERMINE VALIDATION-BASED CLASSIFICATION THRESHOLD
# ============================================================
#
# Our original reported result used threshold 0.5.
#
# For the final pipeline we can also calculate the best accuracy
# threshold on the held-out validation predictions. This is a
# legitimate model-selection step because test labels remain
# completely hidden.
# ============================================================

thresholds = np.linspace(
    0.10,
    0.90,
    161
)

threshold_accs = []

for threshold in thresholds:

    pred = (
        p_val >= threshold
    ).astype(int)

    threshold_accs.append(
        accuracy_score(
            y_val,
            pred
        )
    )

best_threshold_idx = int(
    np.argmax(threshold_accs)
)

FINAL_THRESHOLD = float(
    thresholds[best_threshold_idx]
)

best_threshold_accuracy = float(
    threshold_accs[best_threshold_idx]
)

print(
    "\nValidation threshold selection:"
)

print(
    "Threshold used in original Exp7:",
    0.50
)

print(
    "Best validation threshold:",
    FINAL_THRESHOLD
)

print(
    "Validation accuracy at best threshold:",
    f"{best_threshold_accuracy*100:.4f}%"
)


# ============================================================
# 8. CREATE A FRESH FULL-DATA MODEL
# ============================================================

set_seed(SEED)

final_model = TokenTfidfFusionModel(
    vocab_size=VOCAB_SIZE,
    pad_id=PAD_ID,
    tfidf_dim=TFIDF_SVD_DIM,
    token_latent_dim=TOKEN_LATENT_DIM,
    fused_latent_dim=FUSED_LATENT_DIM
).to(DEVICE)

final_num_params = sum(
    p.numel()
    for p in final_model.parameters()
    if p.requires_grad
)

print(
    "\nFinal model parameters:",
    f"{final_num_params:,}"
)


# ============================================================
# 9. FULL-DATA DATASET / DATALOADER
# ============================================================

full_dataset = FusionDataset(
    X_full_tokens,
    X_full_svd,
    labels,
    full_lengths
)

full_loader = DataLoader(
    full_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


# ============================================================
# 10. FULL-DATA LOSS + OPTIMIZER
# ============================================================

full_n0 = np.sum(
    labels == 0
)

full_n1 = np.sum(
    labels == 1
)

full_pos_weight = (
    full_n0 /
    full_n1
)

final_criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        full_pos_weight,
        dtype=torch.float32,
        device=DEVICE
    )
)

final_optimizer = torch.optim.AdamW(
    final_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print(
    "Full-data class 0:",
    full_n0
)

print(
    "Full-data class 1:",
    full_n1
)

print(
    "Full-data positive weight:",
    full_pos_weight
)


# ============================================================
# 11. TRAIN FINAL MODEL ON ALL LABELLED DATA
# ============================================================

print(
    "\nTraining final model on ALL labelled data..."
)

final_start = time.time()

final_training_history = []

for epoch in range(
    1,
    FINAL_EPOCHS + 1
):

    final_model.train()

    running_loss = 0.0
    total = 0

    for batch in full_loader:

        tokens = batch["tokens"].to(
            DEVICE
        )

        tfidf_batch = batch["tfidf"].to(
            DEVICE
        )

        y = batch["label"].to(
            DEVICE
        )

        final_optimizer.zero_grad()

        logits, z = final_model(
            tokens,
            tfidf_batch
        )

        loss = final_criterion(
            logits,
            y
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            final_model.parameters(),
            max_norm=1.0
        )

        final_optimizer.step()

        batch_size = len(y)

        running_loss += (
            loss.item() *
            batch_size
        )

        total += batch_size

    epoch_loss = (
        running_loss /
        total
    )

    final_training_history.append({
        "epoch": epoch,
        "train_loss": epoch_loss
    })

    print(
        f"Epoch {epoch:02d}/{FINAL_EPOCHS} | "
        f"Train Loss: {epoch_loss:.4f}"
    )


final_training_time = (
    time.time() -
    final_start
) / 60

print(
    f"\nFinal training completed in "
    f"{final_training_time:.2f} min"
)


# ============================================================
# 12. BUILD TEST DATALOADER
# ============================================================

test_tensor_tokens = torch.tensor(
    X_test_tokens,
    dtype=torch.long
)

test_tensor_svd = torch.tensor(
    X_test_svd,
    dtype=torch.float32
)

test_dataset = TensorDataset(
    test_tensor_tokens,
    test_tensor_svd
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


# ============================================================
# 13. PREDICT TEST SET
# ============================================================

print(
    "\nPredicting test set..."
)

final_model.eval()

test_probabilities = []
test_embeddings = []

with torch.no_grad():

    for tokens, tfidf_batch in test_loader:

        tokens = tokens.to(
            DEVICE
        )

        tfidf_batch = tfidf_batch.to(
            DEVICE
        )

        logits, z = final_model(
            tokens,
            tfidf_batch
        )

        probs = torch.sigmoid(
            logits
        )

        test_probabilities.append(
            probs.cpu().numpy()
        )

        test_embeddings.append(
            z.cpu().numpy()
        )


test_probabilities = np.concatenate(
    test_probabilities
)

test_embeddings = np.concatenate(
    test_embeddings
)

print(
    "Test probabilities:",
    test_probabilities.shape
)

print(
    "Test embeddings:",
    test_embeddings.shape
)


# ============================================================
# 14. CONVERT PROBABILITIES → A/B LABELS
# ============================================================

test_predictions_numeric = (
    test_probabilities >= FINAL_THRESHOLD
).astype(int)

test_predictions = np.where(
    test_predictions_numeric == 0,
    "A",
    "B"
)

print(
    "\nPredicted class counts:"
)

print(
    pd.Series(
        test_predictions
    ).value_counts()
)


# ============================================================
# 15. CONSTRUCT / VERIFY submission.csv
# ============================================================

# Prefer the official sample_submission.csv because the project
# specification explicitly recommends preserving its ID column
# and overwriting the prediction column.

if os.path.exists(
    SAMPLE_SUBMISSION_FILE
):

    submission = pd.read_csv(
        SAMPLE_SUBMISSION_FILE
    )

    print(
        "\nLoaded sample submission."
    )

    print(
        "Sample columns:",
        list(submission.columns)
    )

    assert list(
        submission.columns
    ) == [
        "id",
        "label"
    ], (
        "Unexpected sample submission columns."
    )

    assert len(
        submission
    ) == 3000, (
        "Sample submission does not contain 3000 rows."
    )

    # Verify ID/order against test.json.
    assert submission["id"].tolist() == test_ids, (
        "Sample submission IDs/order do not match test.json."
    )

    submission["label"] = test_predictions

else:

    submission = pd.DataFrame({
        "id": test_ids,
        "label": test_predictions
    })


# ============================================================
# 16. FINAL SUBMISSION SANITY CHECKS
# ============================================================

assert list(
    submission.columns
) == [
    "id",
    "label"
]

assert len(
    submission
) == 3000

assert submission["id"].is_unique

assert (
    submission["id"].tolist()
    == test_ids
)

assert set(
    submission["label"].unique()
).issubset(
    {"A", "B"}
)

assert (
    submission["label"].isna().sum()
    == 0
)

SUBMISSION_FILE = (
    "/content/submission_exp7.csv"
)

submission.to_csv(
    SUBMISSION_FILE,
    index=False
)

print(
    "\n============================================================"
)

print(
    "FINAL SUBMISSION CREATED"
)

print(
    "============================================================"
)

print(
    "File:",
    SUBMISSION_FILE
)

print(
    "Rows:",
    len(submission)
)

print(
    "Columns:",
    list(submission.columns)
)

print(
    "\nFirst 10 rows:"
)

display(
    submission.head(10)
)

print(
    "\nClass distribution:"
)

print(
    submission["label"].value_counts()
)


# ============================================================
# 17. SAVE FINAL MODEL / REPORT / TEST EMBEDDINGS
# ============================================================

FINAL_OUTPUT_DIR = Path(
    "/content/exp7_final"
)

FINAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


torch.save(
    {
        "model_state_dict":
            final_model.state_dict(),

        "config": {

            "seed":
                SEED,

            "max_len":
                MAX_LEN,

            "token_embed_dim":
                TOKEN_EMBED_DIM,

            "cnn_channels":
                CNN_CHANNELS,

            "kernels":
                KERNELS,

            "token_latent_dim":
                TOKEN_LATENT_DIM,

            "tfidf_max_features":
                TFIDF_MAX_FEATURES,

            "tfidf_min_df":
                TFIDF_MIN_DF,

            "tfidf_svd_dim":
                TFIDF_SVD_DIM,

            "fused_latent_dim":
                FUSED_LATENT_DIM,

            "final_epochs":
                FINAL_EPOCHS,

            "final_threshold":
                FINAL_THRESHOLD
        }
    },

    FINAL_OUTPUT_DIR /
    "exp7_final_model.pt"
)


np.savez_compressed(
    FINAL_OUTPUT_DIR /
    "exp7_test_embeddings.npz",

    test_embeddings=test_embeddings,

    test_probabilities=test_probabilities,

    test_ids=np.asarray(test_ids)
)


pd.DataFrame(
    final_training_history
).to_csv(
    FINAL_OUTPUT_DIR /
    "final_training_history.csv",
    index=False
)


submission.to_csv(
    FINAL_OUTPUT_DIR /
    "submission_exp7.csv",
    index=False
)


final_report = {

    "experiment":
        "Exp7 Tokens + TF-IDF",

    "validation_neural_accuracy":
        float(best_val_result["accuracy"]),

    "validation_neural_auc":
        float(best_val_result["auc"]),

    "validation_svm_accuracy":
        float(svm_acc),

    "validation_svm_auc":
        float(svm_auc),

    "validation_lr_accuracy":
        float(lr_acc),

    "validation_lr_auc":
        float(lr_auc),

    "validation_best_threshold":
        FINAL_THRESHOLD,

    "validation_accuracy_at_best_threshold":
        best_threshold_accuracy,

    "full_train_documents":
        len(sequences),

    "test_documents":
        len(test_sequences),

    "final_epochs":
        FINAL_EPOCHS,

    "final_training_minutes":
        final_training_time,

    "final_num_parameters":
        final_num_params,

    "predicted_A":
        int(np.sum(test_predictions == "A")),

    "predicted_B":
        int(np.sum(test_predictions == "B"))
}


with open(
    FINAL_OUTPUT_DIR /
    "exp7_final_report.json",
    "w"
) as f:

    json.dump(
        final_report,
        f,
        indent=2
    )


# ============================================================
# 18. ZIP FINAL EXPERIMENT ARTIFACTS
# ============================================================

import shutil

final_zip = shutil.make_archive(
    "/content/exp7_final",
    "zip",
    FINAL_OUTPUT_DIR
)

print(
    "\nFinal artifact ZIP:",
    final_zip
)

print(
    "\nIMPORTANT:"
)

print(
    "Upload ONLY submission_exp7.csv to Kaggle."
)

print(
    "Keep exp7_final.zip/model/report for the project record."
)
